# Conditional constraints with Choco-solver

*Adapted for almond and Scala from the
[Conditionnal_constraint](https://github.com/chocoteam/notebooks/blob/master/Conditionnal_constraint.ipynb)
notebook of [chocoteam/notebooks](https://github.com/chocoteam/notebooks), originally written for a
Java kernel and Choco-solver 4.10. Copyright (c) 2018, chocoteam, distributed under the
[BSD 3-Clause License](https://github.com/chocoteam/notebooks/blob/master/LICENSE).*

See also [Constraint programming with Choco-solver](intro.ipynb) for an introduction.

In this notebook, we are going to see how a constraint can be activated under a variable-based
condition (i.e., another constraint). The small problem we are going to solve is the following:

$$
f(a,b)=\begin{cases}
  a\times b, & \text{if $a>b$},\\
  a+b, & \text{otherwise}.
\end{cases}
$$

There are three ways to handle such a conditional constraint in Choco.

In [1]:
import $ivy.`org.choco-solver:choco-solver:6.0.2`
import org.chocosolver.solver.Model
import org.chocosolver.solver.variables.{BoolVar, IntVar}

import $ivy.$
import org.chocosolver.solver.Model
import org.chocosolver.solver.variables.{BoolVar, IntVar}

Each approach below creates the same variables, then links them in its own way. This helper creates
the model and the variables, lets each approach post its constraints, then enumerates all the
solutions and prints a few statistics:

In [2]:
def solveAll(name: String)(postConstraints: (Model, IntVar, IntVar, IntVar) => Unit): Unit = {
  val model = new Model(name)
  val a = model.intVar("A", 0, 3)
  val b = model.intVar("B", 0, 3)
  val f = model.intVar("F", 0, 10)

  postConstraints(model, a, b, f)

  val solver = model.getSolver
  solver.printShortFeatures()
  while (solver.solve())
    println(s"A=${a.getValue}, B=${b.getValue}, F=${f.getValue}")
  solver.printShortStatistics()
}

defined function solveAll

## First approach: `ifThenElse`

In the first approach, we are going to use an `ifThenElse` constraint that takes three constraints
as parameters. The first constraint is the condition, the second is the one that is satisfied when
the condition holds, the third is the one that is satisfied when the condition doesn't hold.

In [3]:
solveAll("1st try") { (model, a, b, f) =>
  model.ifThenElse(
    model.arithm(a, ">", b),        // if A > B
    model.times(a, b, f),           // then F = A * B
    model.arithm(a, "+", b, "=", f) // else F = A + B
  )
}

Model[1st try], 9 variables, 5 constraints, building time: 0.028s, w/o user-defined search strategy, w/o complementary search strategy
A=0, B=1, F=1
A=0, B=2, F=2
A=0, B=3, F=3
A=1, B=1, F=2
A=1, B=2, F=3
A=1, B=3, F=4
A=2, B=3, F=5
A=3, B=3, F=6
A=0, B=0, F=0
A=2, B=2, F=4
A=1, B=0, F=0
A=2, B=0, F=0
A=3, B=0, F=0
A=2, B=1, F=2
A=3, B=1, F=3
A=3, B=2, F=6
Model[1st try], 16 Solutions, Resolution time 0.024s, 35 Nodes (1,440.8 n/s), 39 Backtracks, 0 Backjumps, 4 Fails, 0 Restarts, 120 Propagations


In that approach, some additional variables and constraints are introduced to make sure the relation
holds. It relies on reification, which attaches a boolean variable to a constraint, set to `true`
when the constraint is satisfied, `false` otherwise (and vice versa). In this approach, the
decomposition is automatically done by the model on the call to `ifThenElse`.

## Second approach: using expressions

Variables can be combined together to form an expression. An expression can be arithmetical,
logical or relational. Once declared, any expression can be turned into an integer variable (for
arithmetical expressions), a boolean variable (for logical expressions) or a constraint (for
relational ones). The latter can either be posted as a combination of constraints or as a table
constraint. In that case, the tuples are generated from the expression (this may not be "free" in
terms of time and space).

Note that in Scala, `eq` is also the reference equality method that every object inherits from
`AnyRef`. When given an expression, `f.eq(…)` still picks Choco's more specific `eq` overload.

In [4]:
solveAll("2nd try") { (model, a, b, f) =>
  f.eq(          // F is equal to …
    a.gt(b).ift( // if A > B
      a.mul(b),  // then A * B
      a.add(b)   // else A + B
    )
  ).decompose()  // decompose the expression
  //).extension() // generate a table constraint
    .post()      // post the constraint
}

Model[2nd try], 13 variables, 8 constraints, building time: 0.011s, w/o user-defined search strategy, w/o complementary search strategy
A=0, B=1, F=1
A=1, B=1, F=2
A=0, B=2, F=2
A=1, B=2, F=3
A=0, B=3, F=3
A=1, B=3, F=4
A=2, B=3, F=5
A=3, B=3, F=6
A=0, B=0, F=0
A=2, B=2, F=4
A=1, B=0, F=0
A=2, B=0, F=0
A=3, B=0, F=0
A=2, B=1, F=2
A=3, B=1, F=3
A=3, B=2, F=6
Model[2nd try], 16 Solutions, Resolution time 0.014s, 37 Nodes (2,686.0 n/s), 43 Backtracks, 0 Backjumps, 6 Fails, 0 Restarts, 245 Propagations


In this approach again, the model automatically decomposes the relation into constraints and
introduces some variables. One may replace `.decompose()` by `.extension()` and note that no
constraint or variable is added:

In [5]:
solveAll("2nd try, as a table") { (model, a, b, f) =>
  f.eq(a.gt(b).ift(a.mul(b), a.add(b))).extension().post()
}

Model[2nd try, as a table], 3 variables, 1 constraints, building time: 0.006s, w/o user-defined search strategy, w/o complementary search strategy
A=0, B=0, F=0
A=1, B=0, F=0
A=2, B=0, F=0
A=3, B=0, F=0
A=0, B=1, F=1
A=1, B=1, F=2
A=2, B=1, F=2
A=3, B=1, F=3
A=0, B=2, F=2
A=1, B=2, F=3
A=2, B=2, F=4
A=3, B=2, F=6
A=0, B=3, F=3
A=1, B=3, F=4
A=2, B=3, F=5
A=3, B=3, F=6
Model[2nd try, as a table], 16 Solutions, Resolution time 0.007s, 31 Nodes (4,611.8 n/s), 31 Backtracks, 0 Backjumps, 0 Fails, 0 Restarts, 28 Propagations


## Third approach: DIY

In that approach, the condition is reified with a boolean variable, which is then used to activate
the two other constraints.

In Java, the negation of a `BoolVar` is written `b.not()`. From Scala 2, `not()` resolves to the
default method of the `ReExpression` interface instead, which returns an expression rather than a
`BoolVar`, so we use the equivalent `model.boolNotView(b)`.

In [6]:
solveAll("3rd try") { (model, a, b, f) =>
  val cond: BoolVar = model.arithm(a, ">", b).reify()
  model.times(a, b, f).impliedBy(cond)                         // cond => F = A * B
  model.arithm(a, "+", b, "=", f).impliedBy(model.boolNotView(cond)) // not(cond) => F = A + B
}

Model[3rd try], 5 variables, 3 constraints, building time: 0.001s, w/o user-defined search strategy, w/o complementary search strategy
A=1, B=0, F=0
A=2, B=0, F=0
A=3, B=0, F=0
A=2, B=1, F=2
A=3, B=1, F=3
A=3, B=2, F=6
A=0, B=0, F=0
A=0, B=1, F=1
A=0, B=2, F=2
A=0, B=3, F=3
A=1, B=1, F=2
A=1, B=2, F=3
A=2, B=2, F=4
A=1, B=3, F=4
A=2, B=3, F=5
A=3, B=3, F=6
Model[3rd try], 16 Solutions, Resolution time 0.004s, 31 Nodes (7,153.3 n/s), 31 Backtracks, 0 Backjumps, 0 Fails, 0 Restarts, 53 Propagations


The `A > B` constraint is reified with `cond`, a boolean variable which is then used to activate
the second constraint. Then, the negation of `cond` is used to activate the third constraint.

`impliedBy` only requires the constraint to hold when the boolean is `true`. The original notebook
used `reifyWith` here, which links them both ways: the constraint must then be *violated* when the
boolean is `false`. That forbids $A \times B = F$ whenever $A \le B$, and loses the two solutions
where $A \times B = A + B$ ($A = B = 0$ and $A = B = 2$), leaving 14 solutions instead of 16.

When a constraint is reified with a boolean variable, it should not be posted (that would force the
boolean variable to always be equal to 1). But the boolean variable can be used to reify other
constraints and/or be linked with other variables in other constraints, like `sum` or clauses.